In [ ]:
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from ipywidgets import interact

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Convolutional Networks
## Spatial Structure, Equivariance, and Hierarchical Vision

This lecture explores Convolutional Neural Networks (CNNs):
- **Spatial Inductive Biases**: Locality, translation equivariance, and weight sharing
- **2D Convolution Operation**: Cross-correlation, multi-channel tensors, stride, and padding
- **Spatial Pooling**: Downsampling, translation invariance, and receptive fields
- **Historic Architectures**: LeNet-5 (1998) to AlexNet (2012) and VGG (2014)
- **The Residual Revolution**: ResNet skip connections (He et al. 2016)
- **Implementation**: PyTorch CNNs, Sobel edge filters, and FashionMNIST classification

# Spatial Inductive Biases in Vision
## Locality and Translation Equivariance

Standard feedforward networks treat inputs as unstructured flat vectors $\mathbf{x} \in \mathbb{R}^D$, ignoring pixel grid topology.

Natural images exhibit two fundamental physical properties:
1. **Local Connectivity**: Nearby pixels are strongly correlated; distant pixels are weakly correlated. Local visual motifs (edges, corners) can be detected with small compact filters.
2. **Translation Equivariance**: A feature detector useful at one location is equally useful anywhere else in the image:
   $$f(g \cdot \mathbf{x}) = g \cdot f(\mathbf{x})$$
   Shifting the input image shifts the resulting feature map by the exact same displacement.

Weight sharing enforces this symmetry, drastically reducing parameter counts while improving generalization.

# The 2D Discrete Convolution Operation
## Cross-Correlation & Multi-Channel Tensors

In deep learning libraries, the "convolution" operation is technically **discrete 2D cross-correlation**.

For a single-channel image $\mathbf{X} \in \mathbb{R}^{H \times W}$ and a kernel $\mathbf{K} \in \mathbb{R}^{K_h \times K_w}$:
$$Y_{i, j} = (\mathbf{X} * \mathbf{K})_{i, j} = \sum_{u=0}^{K_h - 1} \sum_{v=0}^{K_w - 1} X_{i+u, j+v} K_{u, v}$$

For an input tensor with $C_\mathrm{in}$ channels and $C_\mathrm{out}$ output feature maps:
$$Y_{k, i, j} = b_k + \sum_{c=1}^{C_\mathrm{in}} \sum_{u=0}^{K_h - 1} \sum_{v=0}^{K_w - 1} X_{c, i+u, j+v} K_{k, c, u, v}$$

- **Padding ($P$)**: Adding zeros along boundaries preserves spatial resolution (`same` padding: $P = (K - 1)/2$).
- **Stride ($S$)**: Moving the kernel by $S > 1$ pixels subsamples the spatial dimensions:
  $$H_\mathrm{out} = \left\lfloor \frac{H_\mathrm{in} - K_h + 2P}{S} \right\rfloor + 1$$

In [ ]:
# Feature Extraction via Handcrafted Convolution Kernels
# Sobel Filters for Horizontal and Vertical Edge Detection
sobel_h = torch.tensor([[-1., -2., -1.],
                        [ 0.,  0.,  0.],
                        [ 1.,  2.,  1.]]).unsqueeze(0).unsqueeze(0)

sobel_v = torch.tensor([[-1.,  0.,  1.],
                        [-2.,  0.,  2.],
                        [-1.,  0.,  1.]]).unsqueeze(0).unsqueeze(0)

print("Sobel Horizontal Kernel Shape:", sobel_h.shape)
print("Sobel Vertical Kernel Shape  :", sobel_v.shape)

In [ ]:
# Convolution Filtering and Edge Detection Helper
def apply_sobel_filter(filter_type='both'):
    # Generate synthetic geometric image
    img = torch.zeros((1, 1, 32, 32))
    img[0, 0, 8:24, 8:24] = 1.0 # White square on black background
    
    edge_h = F.conv2d(img, sobel_h, padding=1)
    edge_v = F.conv2d(img, sobel_v, padding=1)
    
    if filter_type == 'horizontal':
        res = edge_h.squeeze().numpy()
        title = "Horizontal Edges"
    elif filter_type == 'vertical':
        res = edge_v.squeeze().numpy()
        title = "Vertical Edges"
    else:
        res = torch.sqrt(edge_h**2 + edge_v**2).squeeze().numpy()
        title = "Magnitude Gradient"
        
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    axes[0].imshow(img.squeeze().numpy(), cmap='gray')
    axes[0].set_title("Input Shape (32x32)")
    axes[0].axis('off')
    
    axes[1].imshow(res, cmap='inferno')
    axes[1].set_title(f"Filtered: {title}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive exploration: Detecting oriented edges via convolution
interact(apply_sobel_filter, filter_type=['both', 'horizontal', 'vertical']);

# Spatial Pooling & Receptive Fields
## Downsampling and Invariance

To build robustness against small translations, convolution layers are paired with **Pooling layers**:

1. **Max Pooling**:
   $$Y_{i, j} = \max_{u, v} X_{i \cdot S + u, j \cdot S + v}$$
   Extracts the strongest feature activation within each patch; introduces local translation invariance.
2. **Average Pooling**:
   Computes spatial mean; commonly used as Global Average Pooling before classification heads.

### Receptive Field Growth
- A single $3 \times 3$ convolution layer has an effective receptive field of $3 \times 3$.
- Stacking two $3 \times 3$ convolutions yields a receptive field of $5 \times 5$ (using $2 \times 3^2 = 18$ parameters vs $5^2 = 25$ for a single $5 \times 5$ filter).
- Successive pooling doubles the receptive field growth rate at every stage.

# Architectural Evolution of CNNs
## From LeNet-5 to Modern Deep Vision

```
LeNet-5 (1998) ---> AlexNet (2012) ---> VGG-16 (2014) ---> ResNet (2016)
 5 layers            8 layers            16 layers          152 layers
 (60k params)        (60M params)        (138M params)      (Residual Blocks)
```

1. **LeNet-5 (LeCun et al. 1998)**:
   Introduced the canonical pattern: `[Conv -> Sigmoid -> AvgPool] -> Linear`.
   <small>📖 *Source: [LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document recognition. Proceedings of the IEEE, 86(11), 2278–2324](https://doi.org/10.1109/5.726791)*</small>

2. **AlexNet (Krizhevsky et al. 2012)**:
   Replaced Sigmoid with ReLU, added Dropout, leveraged GPU parallel acceleration, ignited the deep learning revolution.

3. **VGG (Simonyan & Zisserman 2014)**:
   Replaced large filters with stacks of homogeneous $3 \times 3$ convolutions.

# The Residual Revolution (ResNet)
## Overcoming the Vanishing Gradient Dilemma

As network depth increases beyond 20 layers, standard architectures suffer severe degradation: training error saturates and worsens.

Kaiming He et al. (2016) resolved this degradation with **Residual Connections (Skip Connections)**:
$$\mathbf{z}^{(l+1)} = h\left(\mathbf{z}^{(l)} + \mathcal{F}(\mathbf{z}^{(l)}, \mathbf{W}^{(l)})\right)$$

```
Input z --------+---------------------------------> (+) ---> Output z_next
                |                                    ^
                +---> [ Weight ] ---> [ ReLU ] ------+
```

<small>📖 *Source: [He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. CVPR, 770–778](https://doi.org/10.1109/CVPR.2016.90)*</small>

> 💡 **Identity Gradient Highway:** During backpropagation, the gradient decomposes as:

$$\frac{\partial E}{\partial \mathbf{z}^{(l)}} = \frac{\partial E}{\partial \mathbf{z}^{(l+1)}} \left(\mathbf{I} + \frac{\partial \mathcal{F}}{\partial \mathbf{z}^{(l)}}\right)$$

The identity term $\mathbf{I}$ allows gradients to flow directly to early layers without attenuation!

In [ ]:
# PyTorch Residual Block Implementation
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)

print("ResidualBlock defined successfully.")

In [ ]:
# Complete CNN Architecture with Residual Connectivity
class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.res_block = ResidualBlock(16)
        self.pool = nn.MaxPool2d(2, 2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 14 * 14, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.init_conv(x)
        x = self.res_block(x)
        x = self.pool(x)
        return self.classifier(x)

model_cnn = MiniResNet(num_classes=10)
print(f"Instantiated MiniResNet with {sum(p.numel() for p in model_cnn.parameters())} parameters.")

In [ ]:
# Training step verification on FashionMNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST(root='../../shared/data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=0.003)

model_cnn.train()
for batch_idx, (imgs, lbls) in enumerate(train_loader):
    optimizer.zero_grad()
    loss = criterion(model_cnn(imgs), lbls)
    loss.backward()
    optimizer.step()
    if batch_idx >= 30: # Fast demo run
        break

print(f"FashionMNIST CNN 30-step Training Loss: {loss.item():.4f}")

# Summary: Convolutional Networks

In this lecture, we explored deep vision architectures:

1. **Spatial Inductive Biases**:
   - Locality and translation equivariance justify localized weight sharing, eliminating parameter explosion.

2. **Convolution & Pooling**:
   - 2D Cross-correlation extracts multi-channel spatial motifs across receptive fields.
   - Spatial max/average pooling produces translation invariance and downsamples spatial dimensions.

3. **Residual Connections**:
   - ResNet skip connections $\mathbf{z} + \mathcal{F}(\mathbf{z})$ construct identity gradient highways, enabling optimization of ultra-deep networks (100+ layers).